# Generic Transcription Adapter

> The generic (tool-agnostic) transcription adapter — cache-check, invoke the bound tool's pure-compute `transcribe`, persist. Reused across every tool capability satisfying `TranscriptionToolProtocol` (whisper, voxtral, ...), exactly as `GenericGraphStorageAdapter` is reused across graph backends.

**Native-surface model (stage 8 / PILLAR 1c):** the TOOL is pure compute; this adapter adds the cache + persistence bookends + the per-call `force` control. It is the graph-generic pattern PLUS the storage bookends the graph case did not need (the graph tool IS its own store).

In [ ]:
#| default_exp generic

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import os
from pathlib import Path
from typing import Any, Union
from uuid import uuid4

from cjm_substrate.core.wire import get_call_envelope
from cjm_substrate.utils.hashing import hash_file, hash_bytes, hash_dict_canonical

from cjm_capability_primitives.transcription import TranscriptionResult
from cjm_transcription_adapter_interface.adapter import TranscriptionAdapter
from cjm_transcription_adapter_interface.storage import TranscriptionStorage

In [ ]:
#| export
class GenericTranscriptionAdapter(TranscriptionAdapter):
    """Generic transcription adapter: cache-check -> pure-compute tool -> persist.

    Works against ANY tool satisfying `TranscriptionToolProtocol`. The bookends:

      1. cache check (audio_path + audio_hash + config_hash) BEFORE invoking the
         tool, so a hit never loads the model;
      2. the tool's pure-compute `transcribe` on a miss / forced call;
      3. `save_with_logging` (upsert by audio_path + config_hash).

    `config_hash` reuses `hash_dict_canonical(get_current_config())` — the SAME
    canonical hash the fused-era plugins used — so a migrated tool keeps hitting
    rows written before the split. `force` rides `CallEnvelope.control` (not a
    task kwarg, keeping `transcribe(audio)` pure). Storage lives at
    `<CAPABILITY_DATA_DIR>/transcriptions.db`; the substrate injects the
    per-capability `CAPABILITY_DATA_DIR` at spawn (it owns the data-dir convention),
    so the adapter neither hard-codes a path nor asks the tool for one.
    """

    def _get_storage(self) -> TranscriptionStorage:
        """Lazily open the per-capability cache DB under the substrate-injected
        CAPABILITY_DATA_DIR (created at worker spawn)."""
        if getattr(self, "_storage", None) is None:
            db_path = os.path.join(os.environ["CAPABILITY_DATA_DIR"], "transcriptions.db")
            self._storage = TranscriptionStorage(db_path)
        return self._storage

    def transcribe(
        self,
        audio: Union[str, Path],  # Path to MODEL-READY audio (converted upstream)
        **kwargs,                 # Provenance (job_id / source_*_time) + tool options
    ) -> TranscriptionResult:     # Typed transcription output
        """Cache-check, invoke the bound tool's pure-compute `transcribe`, persist."""
        audio_path = str(audio)
        audio_hash = hash_file(audio_path)
        config_hash = hash_dict_canonical(self.tool.get_current_config())

        env = get_call_envelope()
        force = bool(env.control.get("force")) if env is not None else False

        storage = self._get_storage()
        if not force:
            cached = storage.get_cached(audio_path, audio_hash, config_hash)
            if cached is not None:
                return TranscriptionResult(
                    text=cached.text, confidence=None,
                    segments=cached.segments, metadata=cached.metadata,
                )

        # Pure compute on the tool (cache miss / forced).
        result = self.tool.transcribe(audio, **kwargs)

        # Persist. job_id rides the provenance kwarg channel, falling back to the
        # envelope's queue job id, then a fresh uuid (the row id is write-only
        # provenance; the cache key does not include it).
        text_hash = hash_bytes(result.text.encode())
        job_id = kwargs.get("job_id") or (env.job_id if env is not None else None) or str(uuid4())
        storage.save_with_logging(
            job_id=job_id, audio_path=audio_path, audio_hash=audio_hash,
            config_hash=config_hash, text=result.text, text_hash=text_hash,
            segments=result.segments, metadata=result.metadata,
        )
        return result

In [ ]:
# Cache-bookends test: a fake PURE-COMPUTE tool; miss -> compute + persist,
# hit -> NO recompute, force (via CallEnvelope.control) -> bypass + recompute.
import os, tempfile
from cjm_substrate.core.wire import CallEnvelope, set_call_envelope, reset_call_envelope

os.environ["CAPABILITY_DATA_DIR"] = tempfile.mkdtemp(prefix="cjm_gta_test_")

class _FakeTool:
    def __init__(self): self.calls = 0
    def get_current_config(self): return {"model": "fake-1"}
    def transcribe(self, audio, **kwargs) -> TranscriptionResult:
        self.calls += 1
        return TranscriptionResult(text=f"compute:{self.calls}",
                                   metadata={"src": kwargs.get("source_start_time")})

_f = tempfile.NamedTemporaryFile(suffix=".wav", delete=False); _f.write(b"audiobytes"); _f.close()

tool = _FakeTool()
ad = GenericTranscriptionAdapter(tool)

# 1. cache MISS -> compute (calls=1) + persist
r1 = ad.transcribe(_f.name, job_id="j1", source_start_time=1.0)
assert tool.calls == 1 and r1.text == "compute:1"

# 2. cache HIT -> no recompute, same text
r2 = ad.transcribe(_f.name, job_id="j2")
assert tool.calls == 1, f"expected cache hit, calls={tool.calls}"
assert r2.text == r1.text

# 3. force via CallEnvelope.control -> bypass cache, recompute (calls=2)
_tok = set_call_envelope(CallEnvelope(job_id="j3", control={"force": True}))
try:
    r3 = ad.transcribe(_f.name)
finally:
    reset_call_envelope(_tok)
assert tool.calls == 2, f"force should bypass cache, calls={tool.calls}"
print("GenericTranscriptionAdapter cache/force bookends OK")

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()